# Test GPT access through AgentRouter

The GPT companion to `test_claude_connection.ipynb`. No cell prints the API key.

GPT reaches AgentRouter differently from Claude, so each difference is probed on its own:

- **Transport.** GPT goes over the gateway's OpenAI-compatible HTTP API at `{base_url}/v1`; no local CLI is required. Claude instead shells out to the Claude Code binary.
- **Client gate.** AgentRouter authenticates the *client* as well as the key. It answers requests carrying a supported CLI's `User-Agent` and rejects every other one with HTTP 401 `unauthorized_client_error`, even when the key is valid — so a default `python-httpx` request is refused.
- **Wire format.** The gateway may speak chat-completions or responses, so the project attempts both.

Expected configuration:

- Token source: `https://agentrouter.org/console/token`
- Base URL: `https://agentrouter.org`
- Model: `gpt-5.6-sol`
- Local key file: project-root `.env` containing `AGENTROUTER_API_KEY=...`

In [ ]:
from __future__ import annotations

import getpass
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Fail here, with a message that names the fix, rather than deep inside an import chain.
# The project and its dependencies live in the project .venv; no other interpreter has them.
if not Path(sys.prefix).is_relative_to(PROJECT_ROOT):
    raise RuntimeError(
        f"This notebook is running on {sys.executable}, which is not the project .venv.\n"
        "Select the kernel 'Python (semantic_text2sql_v4 .venv)' — or "
        f"{PROJECT_ROOT}/.venv/bin/python — and run again."
    )

import httpx

BASE_URL = "https://agentrouter.org"
MODEL = "gpt-5.6-sol"
ENV_FILE = PROJECT_ROOT / ".env"
PROBE = "Reply exactly: GPT_OK"
print({"project_root": str(PROJECT_ROOT), "base_url": BASE_URL, "model": MODEL})

In [ ]:
def load_env() -> list[str]:
    """Load the project's .env so the end-to-end cell sees the same paths the app does."""
    names: list[str] = []
    if not ENV_FILE.is_file():
        return names
    for line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        os.environ.setdefault(name.strip(), value.strip().strip("\"'"))
        names.append(name.strip())
    return names


def load_key() -> str:
    key = os.environ.get("AGENTROUTER_API_KEY", "").strip()
    if not key:
        key = getpass.getpass("AgentRouter API key (hidden): ").strip()
    if not key:
        raise RuntimeError("No AgentRouter key was provided.")
    return key


ENV_NAMES = load_env()
API_KEY = load_key()
os.environ["AGENTROUTER_API_KEY"] = API_KEY
os.environ.setdefault("AGENTROUTER_BASE_URL", BASE_URL)
print(
    json.dumps(
        {
            "env_names_loaded": ENV_NAMES,
            "key_loaded": True,
            "starts_with_sk": API_KEY.startswith("sk-"),
            "contains_whitespace": any(char.isspace() for char in API_KEY),
            "key_length": len(API_KEY),
        },
        indent=2,
    )
)

## 1. Client gate

Same key, same body, same endpoint — only the `User-Agent` differs. A 401 on the first row and a 200 on the second means the gateway is gating on the client, not on the key. That distinction matters because both failures report HTTP 401 and only the body separates them.

In [ ]:
from semantic_text2sql.llm import AGENTROUTER_USER_AGENT


def probe(path: str, user_agent: str) -> dict[str, object]:
    payload: dict[str, object] = (
        {"model": MODEL, "input": PROBE, "store": False}
        if path.endswith("/responses")
        else {"model": MODEL, "messages": [{"role": "user", "content": PROBE}]}
    )
    try:
        response = httpx.post(
            f"{BASE_URL}{path}",
            headers={
                "authorization": f"Bearer {API_KEY}",
                "content-type": "application/json",
                "user-agent": user_agent,
            },
            json=payload,
            timeout=180,
        )
    except httpx.HTTPError as exc:
        return {"path": path, "user_agent": user_agent, "status": None, "detail": f"{type(exc).__name__}: {exc}"}
    return {
        "path": path,
        "user_agent": user_agent,
        "status": response.status_code,
        "detail": response.text[:200].strip() if response.status_code >= 400 else "",
    }


gate = [
    probe("/v1/chat/completions", f"python-httpx/{httpx.__version__}"),
    probe("/v1/chat/completions", AGENTROUTER_USER_AGENT),
]
print(json.dumps(gate, indent=2))

## 2. Wire formats

`AgentRouterCodexModel` tries `/v1/chat/completions` first and falls back to `/v1/responses` only on a status in its wrong-wire-format set. Both are probed here so a later failure can be attributed to the right one.

In [ ]:
wire = [probe(path, AGENTROUTER_USER_AGENT) for path in ("/v1/chat/completions", "/v1/responses")]
print(json.dumps(wire, indent=2))

## 3. The project's own GPT client

The two checks above use hand-rolled requests. This one exercises `AgentRouterCodexModel` — the class `/api/chat` actually calls — plus the `AgentRouterModel` dispatch that decides whether a model name goes to Claude or to GPT. A pass here and a failure upstream means the bug is not in the transport.

In [ ]:
from semantic_text2sql.llm import (
    AgentRouterClaudeModel,
    AgentRouterCodexModel,
    AgentRouterModel,
    ModelError,
    codex_cli_path,
)

codex = AgentRouterCodexModel(API_KEY, BASE_URL)
router = AgentRouterModel(AgentRouterClaudeModel(API_KEY, BASE_URL), codex)


async def client_test() -> dict[str, object]:
    try:
        text, usage = await codex.complete_detailed(MODEL, PROBE)
    except ModelError as exc:
        return {"connected": False, "error": str(exc)}
    return {
        "connected": "GPT_OK" in text,
        "result": text[:200],
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cache_read_tokens": usage.cache_read_tokens,
    }


client_result = await client_test()
print(
    json.dumps(
        {
            "user_agent": codex.user_agent,
            "codex_cli_on_path": codex_cli_path(),
            "dispatches_gpt_to_codex": router._select(MODEL) is codex,
            "dispatches_claude_to_claude": router._select("claude-opus-5") is not codex,
            **client_result,
        },
        indent=2,
    )
)

## 4. Catalog

`/api/models` is what the web UI lists. A model with `configured: false` is not selectable in the dropdown, so an entry that cannot serve requests explains a GPT option the user never sees. `unavailable_reason` carries the cause.

In [ ]:
from fastapi.testclient import TestClient

from semantic_text2sql.api import create_app

client = TestClient(create_app())
catalog = client.get("/api/models").json()
print(json.dumps(catalog, indent=2))

## 5. End-to-end generation

`/api/chat` is the only generation entry point this app exposes — there is no `/api/generate`, so a request to that path returns 404 regardless of provider. This cell drives a real question through GPT and separates two outcomes that look alike from the UI: an unreachable model (`termination_reason: model_error`) versus a reachable model whose SQL the validator rejected (`attempt_limit`).

In [ ]:
QUESTION = "How many schools are there in Alameda county?"

available = [item["db_id"] for item in client.get("/api/databases").json() if item["configured"]]
preferred = os.environ.get("TEXT2SQL_NOTEBOOK_DB", "california_schools")
db_id = preferred if preferred in available else (available[0] if available else preferred)
print({"db_id": db_id, "configured_databases": available[:8]})

response = client.post(
    "/api/chat",
    json={
        "session_id": "gpt-access-check",
        "db_id": db_id,
        "message": QUESTION,
        "provider": "agentrouter",
        "model": MODEL,
        "execute": True,
        "max_rows": 5,
    },
)
chat = response.json() if response.status_code == 200 else {}
generation = chat.get("generation") or {}
print("HTTP", response.status_code)
print(
    json.dumps(
        {
            "message": chat.get("message") or response.text[:300],
            "accepted": generation.get("accepted"),
            "termination_reason": generation.get("termination_reason"),
            "model_error": generation.get("model_error"),
            "sql": generation.get("sql"),
            "row_count": generation.get("row_count"),
            "rows": generation.get("rows", [])[:3],
            "attempts": [
                {
                    "n": attempt["number"],
                    "code": attempt["validation"]["code"],
                    "message": attempt["validation"]["message"][:160],
                }
                for attempt in generation.get("attempts", [])
            ],
        },
        indent=2,
        default=str,
    )
)

## Verdict

In [ ]:
lines: list[str] = []

if client_result.get("connected"):
    lines.append("PASS: the project's GPT client reached AgentRouter and returned the expected text.")
else:
    detail = client_result.get("error") or "the reply did not contain GPT_OK"
    lines.append(f"FAIL: the GPT client did not connect: {detail}")

if gate[0]["status"] == 401 and gate[1]["status"] == 200:
    lines.append(
        "NOTE: the gateway rejected a default python-httpx User-Agent and accepted the Codex one, "
        "so AGENTROUTER_USER_AGENT must stay aligned with a client the gateway supports."
    )
elif gate[0]["status"] == gate[1]["status"]:
    lines.append(f"NOTE: the User-Agent did not change the outcome (both {gate[0]['status']}).")

answering = [item["path"] for item in wire if item["status"] == 200]
lines.append(f"Wire formats answering 200: {answering or 'none'}")

gpt_entry = next(
    (item for item in catalog if item["provider"] == "agentrouter" and item["model"] == MODEL), None
)
if gpt_entry is None:
    lines.append(f"FAIL: {MODEL} is absent from /api/models, so the UI cannot offer it.")
elif not gpt_entry["configured"]:
    lines.append(f"FAIL: /api/models marks {MODEL} unavailable: {gpt_entry['unavailable_reason']}")
else:
    lines.append(f"PASS: /api/models offers {MODEL} as selectable.")

if generation.get("accepted"):
    lines.append(
        f"PASS: end-to-end generation accepted SQL and returned {generation.get('row_count')} rows."
    )
elif generation.get("termination_reason") == "model_error":
    lines.append(f"FAIL: end-to-end generation could not reach GPT: {generation.get('model_error')}")
elif generation:
    lines.append(
        "PARTIAL: GPT answered but the validator rejected its SQL "
        f"({generation.get('termination_reason')}). Access is fine; SQL quality is the open issue."
    )
else:
    lines.append("FAIL: /api/chat returned no generation block.")

print("\n".join(lines))